# 01 · Data collection: Disney / 21st Century Fox

This notebook collects, extracts and checks the historical financial and transaction data for The Walt Disney
Company's acquisition of Twenty-First Century Fox, Inc. (21CF). **No valuation modelling happens here.** It
builds and checks the inputs that the existing DCF / Black-Scholes / binomial / Monte Carlo notebook
(`MA_Real_Options_Valuation.ipynb`) will use later.

| Stage | Date | What happened | Primary source |
|---|---|---|---|
| Original agreement | 13-14 Dec 2017 | All-stock offer, 0.2745 Disney shares per 21CF share (~$52.4bn equity value) | Disney 8-K (14 Dec 2017), 1st joint proxy (30 May 2018) |
| Amended agreement | 20 Jun 2018 | $38.00 per share in cash or stock, 50/50 mix, collar (~$71.3bn equity value) | Disney 8-K (21 Jun 2018), **2018 joint proxy (28 Jun 2018)** |
| Closing | 19-20 Mar 2019 | Fox Corporation spun off, then the merger became effective; $51.572626 cash or 0.4517 shares per remaining 21CF share | New Disney 8-K12B, 21CF 8-Ks (Mar 2019) |
| Purchase accounting | FY2019 | ASC 805 purchase price $69,507m; purchase price allocation | Disney 10-Q Q2 FY2019, 10-K FY2019 |

**Acquired perimeter.** Disney did not buy all of 21CF. Before closing, 21CF moved its news, sports and broadcast
businesses into Fox Corporation ("New Fox") and spun it off. Disney acquired the remainder, called "RemainCo" in
the filings. The dataset therefore holds 21CF consolidated history *and* RemainCo pro forma history; they are
kept apart with the `entity` / `perimeter` columns.

The pipeline is implemented in `src/`. This notebook runs it and shows the results.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebook run from notebooks/
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
print("repository root:", ROOT)

## 1. Run the pipeline

`src/run_pipeline.py` performs these steps in order:

1. `fetch_sources.py`: downloads each filing in `data/reference/filings_to_fetch.csv` from SEC EDGAR into
   `data/raw/sec/`, and the SEC XBRL company-facts JSON into `data/raw/xbrl/`. Sizes and SHA-256 hashes go to
   `data/raw/download_manifest.csv`. Files that already exist are not downloaded again.
2. `sec_text.py`: converts the HTML filings to text and table caches in `data/interim/` (derived, can be rebuilt).
3. `build_historical.py`: writes `historical_financials.csv` and the arithmetic and XBRL checks.
4. `build_transaction.py`: writes `transaction_assumptions.csv`, checking each quote and value against its source.
5. `build_source_register.py`: writes `source_register.csv`.

Set `OFFLINE = False` to download missing files. SEC asks automated clients to identify themselves, so set the
`SEC_USER_AGENT` environment variable to `"Your Name your@email"` before downloading.

In [ ]:
OFFLINE = all((ROOT / "data" / "raw" / "sec").glob("*")) and (ROOT / "data" / "raw" / "download_manifest.csv").exists()
args = [sys.executable, str(ROOT / "src" / "run_pipeline.py")] + (["--offline"] if OFFLINE else [])
res = subprocess.run(args, cwd=ROOT, capture_output=True, text=True, encoding="utf-8")
print("\n".join(l for l in res.stdout.splitlines() if not l.startswith(("text ", "ok "))))
print(res.stderr[-2000:])
assert res.returncode == 0, "pipeline failed - see output above"

## 2. Source register

Every figure in the processed datasets points to a `source_id` listed here. URLs are EDGAR archive locations
built from the accession number, and the SHA-256 identifies the exact file that was parsed.

In [ ]:
src = pd.read_csv(PROC / "source_register.csv")
src.assign(sha256=src.sha256.str[:12])[["source_id", "form", "filing_date", "period_end", "accession",
    "download_status", "records_historical_financials", "records_transaction_assumptions", "sha256"]]

## 3. Fiscal calendars (kept as reported)

* **Disney**: fiscal year ends on the Saturday closest to 30 September. FY2015 was a **53-week** year
  (28 Sep 2014 to 3 Oct 2015). The others are 52 weeks.
* **21CF**: fiscal year ends **30 June**.

So "FY2017" means Oct 2016 to Sep 2017 for Disney but Jul 2016 to Jun 2017 for 21CF, a three-month offset.
Values are **not** calendarised. Each row carries its own `period_start` / `period_end`.

In [ ]:
hf = pd.read_csv(PROC / "historical_financials.csv", low_memory=False, dtype={"source_page": str})
cal = (hf[hf.period_length != "instant"]
       .drop_duplicates(["entity", "fiscal_period"])[["entity", "fiscal_period", "period_start", "period_end", "period_length"]]
       .sort_values(["entity", "period_end"]))
cal

## 4. Historical financials: primary values

`is_primary == True` picks one value per entity, period and line item:

* FY2015-FY2017: the **FY2017 10-K** of each company, the last annual report before the deal was announced,
  so all three years are on one basis. FY2015 balance sheets come from the FY2016 10-K comparatives.
* FY2018 (supplementary): the FY2018 10-K.

Sign convention: as presented. Expenses and cash outflows are negative. Units are in the `unit` column.

In [ ]:
key = ["revenue", "depreciation_amortization", "pretax_income_continuing", "income_tax", "net_income",
       "net_income_attributable_parent", "eps_diluted", "cash", "total_borrowings", "net_debt", "total_assets",
       "equity_attributable_parent", "total_equity", "cfo", "capex", "fcf_cfo_less_capex"]
prim = hf[hf.is_primary & hf.entity.isin(["DIS", "21CF"]) & hf.line_item.isin(key)]
(prim.pivot_table(index="line_item", columns=["entity", "fiscal_period"], values="value", aggfunc="first")
     .reindex(key))

In [ ]:
seg = hf[hf.is_primary & hf.line_item.isin(["segment_revenue", "segment_operating_income", "segment_oibda"])]
seg.pivot_table(index=["entity", "line_item", "segment"], columns="fiscal_period", values="value", aggfunc="first", sort=False)

## 5. Acquired perimeter: 21CF to RemainCo

The RemainCo pro forma statements start from 21CF historical, remove New Fox (presented as discontinued
operations under ASC 205, with *no* carve-out allocation of corporate costs), and add pro forma adjustments.

* `pro_forma_2018_joint_proxy`: RemainCo as described in the **2018 joint proxy** (FY2015-FY2017, six months to
  Dec 2017, balance sheet at 31 Mar 2018). The proxy's hypothetical "consolidation of Sky" columns are left out
  because 21CF never acquired the rest of Sky.
* `pro_forma_8KA_closing_perimeter`: RemainCo as filed with Disney's **March 2019 8-K/A**. This is the actual
  closing perimeter, with New Fox spun off and the 39% Sky stake sold in October 2018 (FY2016-FY2018, Q1 FY2019,
  balance sheet at 31 Dec 2018). It is primary wherever it covers a period.

In [ ]:
pf = hf[hf.entity.isin(["21CF", "RemainCo", "RemainCo_bridge"]) & hf.basis.str.startswith("pro_forma")
        & hf.line_item.isin(["revenue", "pretax_income_continuing"])]
pf.pivot_table(index=["line_item", "basis", "fiscal_period"], columns="perimeter", values="value", aggfunc="first")

The same period can differ between the two pro forma presentations because 21CF reclassified costs between
operating expenses and SG&A in its FY2018 10-K. Revenue and pretax income agree:

In [ ]:
cmp = hf[(hf.entity == "RemainCo") & hf.fiscal_period.isin(["FY2016", "FY2017"])]
cmp.pivot_table(index=["fiscal_period", "line_item"], columns="basis", values="value", aggfunc="first").dropna()

## 6. Restatements and reclassifications across filings

Each 10-K also presents earlier years. Where a later filing shows a different number for the same line and
period, both values are kept (`basis` = `as_originally_reported` vs `comparative_in_later_filing`). The table
lists every difference found.

In [ ]:
rep = hf[(hf.data_type == "reported") & hf.statement.isin(["income_statement", "balance_sheet", "cash_flow"])]
g = rep.groupby(["entity", "statement", "section", "fiscal_period", "line_item_reported"], dropna=False)
diff = g.filter(lambda x: x.value.nunique() > 1)
(diff.pivot_table(index=["entity", "statement", "section", "fiscal_period", "line_item_reported"], columns="source_id",
                  values="value", aggfunc="first"))

## 7. Transaction terms by deal stage

`measure_basis` keeps headline equity value, headline transaction value (equity plus net debt), accounting
purchase consideration and per-share terms apart. `stage` keeps the original offer, the amended offer, the
closing terms and the purchase accounting apart. Derived rows are recomputed from the rows named in `formula`
and checked against the reported counterpart.

In [ ]:
ta = pd.read_csv(PROC / "transaction_assumptions.csv", dtype={"source_page": str})
headline = ta[ta.measure_basis.isin(["equity_value_headline", "transaction_value_headline", "net_debt_assumed_headline",
                                     "purchase_consideration_accounting", "cash_consideration",
                                     "stock_consideration_accounting", "stock_consideration_implied", "per_share_consideration"])]
headline[["id", "stage", "item", "value", "unit", "measure_basis", "value_type", "source_id", "source_page", "verification_status"]]

In [ ]:
ta[ta.stage == "accounting_fy2019"][["id", "item", "value", "unit", "source_page", "verification_status"]]

## 8. Verification summary

* `arithmetic`: tie-outs inside each statement (assets = liabilities + equity; pretax + tax = net income;
  NCI bridge; segment totals = consolidated revenue).
* `xbrl_cross_check`: each mapped 10-K value is compared with the SEC XBRL fact for the same accession and period,
  a check independent of the HTML parser.
* `transaction_quote` / `transaction_table_value`: the quote is found in the filing text and the stated number
  matches the stored value.
* `transaction_derived`: recomputed figures agree with the reported figure within the rounding of the source.

In [ ]:
vc = pd.read_csv(PROC / "verification_checks.csv")
summary = vc.groupby(["check_type", "status"]).size().unstack(fill_value=0)
display(summary) if "display" in globals() else print(summary)
failures = vc[vc.status.isin(["fail", "mismatch"])]
print(f"failures: {len(failures)}")
assert failures.empty

## 9. What is not here

See `docs/data_collection_report.md` for the full list of limitations and missing items. In short: there is no
RemainCo cash-flow statement or segment split, no calendarisation, RSN standalone data is limited to the
Disney pro forma bridge, and management forecasts and market price data were deliberately not collected at
this stage.